In [1]:
import cv2
import argparse
import numpy as np
import math


def cv_show(name, image):
    cv2.imshow(name, image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


capture = cv2.VideoCapture(0)
if capture.isOpened() is False:
    print("Error opening the camera")
while capture.isOpened():
    ret, frame = capture.read()

    if ret is True:
        img = frame

        # 将图像转换为灰度图像
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # 应用高斯滤波以平滑图像并去除噪声
        blur = cv2.GaussianBlur(gray, (5, 5), 0)

        # 二值化
        ret, thresh = cv2.threshold(blur, 55, 255, cv2.THRESH_BINARY)
        # 应用Canny边缘检测器以检测黑线
        edges = cv2.Canny(thresh, 50, 150, apertureSize=3)
        kernel = np.ones((10, 10), np.uint8)
        dilate = cv2.dilate(edges, kernel, iterations=3)  # 膨胀（反腐蚀）dilate
        # 运行霍夫线变换以检测直线
        lines = cv2.HoughLines(thresh, 1, np.pi / 180, 100)

        # 绘制检测到的线
        if lines is not None:
            for line in lines:
#                 print('find')
                # x1, y1, x2, y2 = line[0]
                rho, theta = line[0]
                a = np.cos(theta)
                b = np.sin(theta)
                x0 = a * rho
                y0 = b * rho
                x1 = int(x0 + 1000 * (-b))
                y1 = int(y0 + 1000 * (a))
                x2 = int(x0 - 1000 * (-b))
                y2 = int(y0 - 1000 * (a))
                cv2.line(thresh, (x1, y1), (x2, y2), (255, 255, 255), 1)
                pos = int(lines[0][0][0] * np.cos(lines[0][0][1]))
#                 print(pos)
        else:
            print('none')

        # 显示图像

        #
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        ret, img = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)  # 超过阈值127部分取所设定的最大值255，否则取0
        edges = cv2.Canny(img, 50, 150, apertureSize=3)
        
        lines = cv2.HoughLines(edges, 1, np.pi / 180, 200)
        if lines is not None:
            for line in lines:
                rho, theta = line[0]
                a = np.cos(theta)
                b = np.sin(theta)
                x0 = a * rho
                y0 = b * rho
                x1 = int(x0 + 1000 * (-b))
                y1 = int(y0 + 1000 * (a))
                x2 = int(x0 - 1000 * (-b))
                y2 = int(y0 - 1000 * (a))
        
                # cv2.line(img, (x1, y1), (x2, y2), (0, 0, 255), 2)
                lineMagnitude = math.sqrt(math.pow((x2 - x1), 2) + math.pow((y2 - y1), 2))
                cv2.line(img, (x1, y1), (x2, y2), 0, 3)  # 255画白线,2粗细
        else:
            True

        cv2.imshow('Grayscale input camera', img)

        if cv2.waitKey(20) & 0xFF == ord('q'):
            break
    else:
        break
capture.release()
cv2.destroyAllWindows()
